# Rust 119: PostgreSQL

Chapter 19, and chapter 18's client discipline lands on a *relational* server. `sqlx` is the async SQL toolkit: a **pool** of connections you clone freely (chapter 18's multiplexed clone, with real sockets behind it), **runtime queries** that return rows you map yourself, **`FromRow`** as the struct-at-the-boundary contract (chapter 17's serde derive, hand-rolled), **transactions that survive `.await` points** (chapter 16's rules applied to a server-side promise), and **`NULL` as `Option`** — chapter 8's optionality, enforced by the schema. Every query in this chapter runs against a **local PostgreSQL server built inside the notebook** (§0) — a std `TcpListener` speaking the real PostgreSQL v3 wire protocol: startup handshake, extended query (`Parse`/`Bind`/`Describe`/`Execute`/`Sync`), `DataRow` decoding, and the `ReadyForQuery` transaction-status byte that `sqlx` checks across awaits. Deterministic, offline, honest.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `19_postgresql_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer arriving from Python and C++. `psycopg` and libpq are old friends; C++ taught you that a database driver is just a socket protocol with types. What is new in Rust: **sqlx checks your SQL against the live server at compile time** in its famous macro mode — and this chapter deliberately teaches the *runtime* mode beside it, because the two share every type; **rows are decoded into your types at the call site** (`row.get("id")` or `FromRow`), so a schema change is a *decode* error, not a mystery; **the transaction is a value** (`Transaction` guards the connection — drop it and the rollback is issued for you); and **`NULL` cannot sneak past the type system**: a nullable column is `Option<T>` in your struct or the decode refuses. The error story is layered: transport errors, SQL errors *with codes* (`42P01`), and decode errors are three different shapes, and §10 teaches you to read which layer failed.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cells are independent — no sandbox, no files touched, so restart-and-run-from-the-top always works.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic you would get. §10 is the catalogue.
- The running domain is the trading desk: a `fills` table with ids, symbols, quantities, and a nullable `note` column. Every API here earns its place on that domain.
- One honesty note runs through the chapter: **the server is local and small**. §0.2 implements the v3 wire protocol — the same protocol a real PostgreSQL speaks — on an ephemeral port; one table shape, real transaction semantics (snapshot at `BEGIN`, restore on rollback), real error codes. The places a full server would differ (concurrent writers, WAL, plan optimization) are marked in §0.2's commentary.


### Contents

- [0. Notebook Conventions: A Local PostgreSQL](#0-notebook-conventions-a-local-postgresql)
- [1. Connecting: Options and the Pool](#1-connecting-options-and-the-pool)
- [2. Queries: fetch_all, fetch_optional, fetch_one](#2-queries-fetch_all-fetch_optional-fetch_one)
- [3. Rows to Structs: FromRow](#3-rows-to-structs-fromrow)
- [4. NULL and Option](#4-null-and-option)
- [5. Transactions Across Await Points](#5-transactions-across-await-points)
- [6. Bind Parameters and SQL Injection](#6-bind-parameters-and-sql-injection)
- [7. The Pool: Sizing, Acquire Timeouts](#7-the-pool-sizing-acquire-timeouts)
- [8. Migrations: Schema as Code](#8-migrations-schema-as-code)
- [9. sqlx in the Ecosystem: Macros, tokio-postgres, Diesel](#9-sqlx-in-the-ecosystem-macros-tokio-postgres-diesel)
- [10. Reading the Compiler: E-Codes](#10-reading-the-compiler-e-codes)
- [11. Mental Model: The Receipts](#11-mental-model-the-receipts)
- [12. Cheat Sheet](#12-cheat-sheet)
- [13. Review](#13-review)

If you are returning to this chapter later, §12 is the one-page summary, §10 the diagnostics catalogue, and §5 the transaction pattern every real service is built on.


## 0. Notebook Conventions: A Local PostgreSQL

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. The standing rules from chapter 16 all hold: no `fn main`, state persists across cells, items are order-independent, one namespace per notebook, first cell that needs a name does the `use`. Four chapter-specific conventions:

- **A cell that wants async creates its runtime and lets it drop** (`Runtime::new()` + `block_on`) — the manual form, because the kernel owns `main`. Pool handles are created *inside* the cell that uses them: a pool dropped at cell end closes its connections, and the desk handles that like any disconnect.
- **The server is local.** §0.2 builds the wire-protocol server on `127.0.0.1:0` — the OS picks a free port — and binds its port for the rest of the notebook. Every pool in every cell dials that port.
- **Items live before the cell that uses them.** The server's helper `fn`s and the `Fill` struct are items; the `pg_port` binding is a *statement* made once in §0.2 and read everywhere after.
- **Types mirror the schema.** The desk's table is `t(id int4, name text)` with `name` nullable — so the Rust side is `i32` and `Option<String>`, and §4 is about why that pairing is not negotiable.

### The crates

`sqlx` (features `runtime-tokio` + `postgres`) is the client: `runtime-tokio` is what makes the tokio connection types exist — chapter 15's feature-flag lesson again. **The `macros` feature is deliberately absent**: `query!`-style compile-time checking needs a live `DATABASE_URL` at build time, which a notebook cannot promise, so §2 teaches the runtime API and §9 explains exactly what the macros would add. `tokio` supplies the runtime each cell builds.


In [ ]:
// 0.1 The chapter's crate cell: run once per kernel session, features included.
:dep sqlx = { version = "0.8", default-features = false, features = ["runtime-tokio", "postgres"] }
:dep tokio = { version = "1", features = ["full"] }

// (without "runtime-tokio" the tokio pool/connection types do not exist; without
// the "macros" feature the query! macro family is absent — that is §9's topic)
println!("crates loaded: sqlx (runtime-tokio, postgres), tokio");


### The local server

A PostgreSQL client speaks **protocol version 3**: after a TCP connect, the client sends a `StartupMessage` (protocol 196608 = "3.0", plus key/value pairs like `user`), the server answers `AuthenticationOk`, a handful of `ParameterStatus` records, and `ReadyForQuery` — whose final byte is the **transaction status** (`I`dle / `T`ransaction). Queries then flow as the **extended query protocol**: the client sends `Parse` (name a statement), `Bind` (name a portal with parameters), `Describe` (ask for the row shape), `Execute` (run the portal), `Sync` (commit the batch); the server answers each and ends every batch with `ReadyForQuery`. Data rows travel as `DataRow` records of typed columns — `int4` arrives as 4 big-endian bytes, `text` as UTF-8, `NULL` as a length of −1 and no bytes at all. §0.2 implements exactly that grammar plus one table (`t`), `BEGIN`/`COMMIT`/`ROLLBACK` with snapshot semantics, and PostgreSQL error codes on failures. Three fidelity notes: `sqlx` sends an `SSLRequest` first and accepts the `N`o reply (real servers with TLS enabled would continue the handshake); statement and portal names are per-connection state the client manages; and there is one table shape — a teaching desk, not a planner.


In [ ]:
// 0.2 The local PostgreSQL desk: a raw std TcpListener speaking protocol v3.
use std::collections::HashMap;
use std::io::{Read, Write};
use std::sync::{Arc, Mutex};

// ---- message builders: [tag][i32 len][body]; len counts its own 4 bytes ----
fn msg(tag: u8, body: &[u8]) -> Vec<u8> {
    let mut out = vec![tag];
    out.extend_from_slice(&((body.len() + 4) as i32).to_be_bytes());
    out.extend_from_slice(body);
    out
}

fn cstr(s: &str) -> Vec<u8> {
    let mut v = s.as_bytes().to_vec();
    v.push(0);
    v
}

fn i32be(n: i32) -> Vec<u8> { n.to_be_bytes().to_vec() }
fn i16be(n: i16) -> Vec<u8> { n.to_be_bytes().to_vec() }

fn authentication_ok() -> Vec<u8> { msg(b'R', &i32be(0)) }

fn parameter_status(k: &str, v: &str) -> Vec<u8> {
    let mut b = cstr(k);
    b.extend(cstr(v));
    msg(b'S', &b)
}

fn ready_for_query(status: u8) -> Vec<u8> { msg(b'Z', &[status, 0]) }

fn command_complete(tag: &str) -> Vec<u8> { msg(b'C', &cstr(tag)) }

fn error_response(code: &str, message: &str) -> Vec<u8> {
    let mut b = vec![b'S'];
    b.extend(cstr("ERROR"));
    b.push(b'V');
    b.extend(cstr("ERROR"));
    b.push(b'C');
    b.extend(cstr(code));
    b.push(b'M');
    b.extend(cstr(message));
    b.push(0);
    msg(b'E', &b)
}

fn parse_complete() -> Vec<u8> { msg(b'1', &[]) }
fn bind_complete() -> Vec<u8> { msg(b'2', &[]) }
fn no_data() -> Vec<u8> { msg(b'n', &[]) }
fn empty_query_response() -> Vec<u8> { msg(b'I', &[]) }

// ---- the desk's one shape: t(id int4 binary, name text) ----
const INT4: i32 = 23;
const TEXT: i32 = 25;

struct Col { name: &'static str, oid: i32, fmt: i16 } // fmt: 0 text, 1 binary

fn int4col(name: &'static str) -> Col { Col { name, oid: INT4, fmt: 1 } }
fn textcol(name: &'static str) -> Col { Col { name, oid: TEXT, fmt: 0 } }

fn row_description(cols: &[Col]) -> Vec<u8> {
    let mut b = i16be(cols.len() as i16);
    for c in cols {
        b.extend(cstr(c.name));
        b.extend(i32be(0)); // table oid
        b.extend(i16be(0)); // attnum
        b.extend(i32be(c.oid));
        b.extend(i16be(if c.oid == INT4 { 4 } else { -1 })); // typlen
        b.extend(i32be(-1)); // typmod
        b.extend(i16be(c.fmt));
    }
    msg(b'T', &b)
}

// a cell is None (NULL) or raw wire bytes (int4 = 4 bytes BE, text = utf8)
type Cell = Option<Vec<u8>>;
type WireRow = Vec<Cell>;

fn data_row(row: &[Cell]) -> Vec<u8> {
    let mut b = i16be(row.len() as i16);
    for cell in row {
        match cell {
            Some(v) => { b.extend(i32be(v.len() as i32)); b.extend(v); }
            None => b.extend(i32be(-1)), // NULL: length -1, no bytes
        }
    }
    msg(b'D', &b)
}

#[derive(Default)]
struct Db {
    rows: Vec<(i32, Option<String>)>, // t(id int4 pk-ish, name text null)
    next_id: i32,
    snapshot: Option<(Vec<(i32, Option<String>)>, i32)>, // taken at BEGIN
}

impl Db {
    fn tx_note(&self, kind: &str) {
        use std::io::Write;
        if let Ok(mut f) = std::fs::OpenOptions::new()
            .create(true).append(true).open("pg_tx.log")
        {
            writeln!(f, "{}", kind).ok();
        }
    }
}

type Shared = Arc<Mutex<Db>>;

/// Run one bound query; return (payload messages, new in_tx).
fn run_query(db: &mut Db, sql: &str, params: &[Option<String>], in_tx: bool) -> (Vec<u8>, bool) {
    let q = sql.trim();
    let ql = q.to_lowercase();
    if q.is_empty() {
        return (empty_query_response(), in_tx);
    }
    let err = |code: &str, m: &str| (error_response(code, m), in_tx);
    let p0 = || params.first().and_then(|p| p.clone());

    if ql == "begin" || ql.starts_with("begin ") {
        if in_tx { return err("25001", "there is already a transaction in progress"); }
        db.snapshot = Some((db.rows.clone(), db.next_id)); // isolation: snapshot
        db.tx_note("BEGIN");
        return (command_complete("BEGIN"), true);
    }
    if ql == "commit" || ql == "commit work" || ql == "end" {
        if !in_tx { return err("25P01", "there is no transaction in progress"); }
        db.snapshot = None; // keep the working set
        db.tx_note("COMMIT");
        return (command_complete("COMMIT"), false);
    }
    if ql.starts_with("rollback") {
        if !in_tx { return err("25P01", "there is no transaction in progress"); }
        if let Some((rows, next)) = db.snapshot.take() {
            db.rows = rows;
            db.next_id = next;
        }
        db.tx_note("ROLLBACK");
        return (command_complete("ROLLBACK"), false);
    }
    if ql == "select 1" || ql == "select 1;" {
        let cols = [int4col("n")];
        let mut out = row_description(&cols);
        out.extend(data_row(&[Some(i32be(1))]));
        out.extend(command_complete("SELECT 1"));
        return (out, in_tx);
    }
    if ql.starts_with("select count(*)") {
        let cols = [int4col("count")];
        let mut out = row_description(&cols);
        out.extend(data_row(&[Some(i32be(db.rows.len() as i32))]));
        out.extend(command_complete("SELECT 1"));
        return (out, in_tx);
    }
    if ql.starts_with("select") {
        if !ql.contains("from t") && !ql.contains("from fills") {
            let table = ql.split_whitespace()
                .skip_while(|w| *w != "from").nth(1).unwrap_or("?").trim_end_matches(';');
            return err("42P01", &format!("relation \"{}\" does not exist", table));
        }
        let cols = [int4col("id"), textcol("name")];
        let filter = if ql.contains("where name = $1") { p0() } else { None };
        let mut rs: Vec<WireRow> = db.rows.iter()
            .filter(|(_, name)| match &filter {
                Some(f) => name.as_deref() == Some(f.as_str()),
                None => true,
            })
            .map(|(id, name)| vec![
                Some(i32be(*id)),
                name.as_ref().map(|n| n.as_bytes().to_vec()),
            ])
            .collect();
        rs.sort_by(|a, b| match (&a[0], &b[0]) {
            (Some(x), Some(y)) => x.cmp(y),
            _ => std::cmp::Ordering::Equal,
        });
        let tag = format!("SELECT {}", rs.len());
        let mut out = row_description(&cols);
        for r in &rs { out.extend(data_row(r)); }
        out.extend(command_complete(&tag));
        return (out, in_tx);
    }
    if ql.starts_with("insert") {
        let id = db.next_id;
        db.next_id += 1;
        db.rows.push((id, p0()));
        return (command_complete("INSERT 0 1"), in_tx);
    }
    if ql.starts_with("update") {
        let nv = p0();
        let mut n = 0;
        for r in db.rows.iter_mut() { r.1 = nv.clone(); n += 1; }
        return (command_complete(&format!("UPDATE {}", n)), in_tx);
    }
    if ql.starts_with("delete") {
        let n = db.rows.len();
        db.rows.clear();
        return (command_complete(&format!("DELETE {}", n)), in_tx);
    }
    err("42601", &format!("syntax error at or near \"{}\"", &q[..q.len().min(24)]))
}

fn returns_rows(sql: &str) -> bool {
    sql.trim().to_lowercase().starts_with("select")
}

fn cstring_at(body: &[u8], from: usize) -> String {
    let mut end = from;
    while end < body.len() && body[end] != 0 { end += 1; }
    String::from_utf8_lossy(&body[from..end]).into_owned()
}

struct Conn {
    statements: HashMap<String, String>,          // name -> sql
    portals: HashMap<String, (String, Vec<Option<String>>)>, // name -> (sql, params)
    in_tx: bool,
}

fn handle(mut stream: std::net::TcpStream, db: Shared) {
    let _ = stream.set_nodelay(true);
    let mut buf: Vec<u8> = Vec::new();
    let mut tmp = [0u8; 16384];

    // ---- startup: possibly SSLRequest first, then StartupMessage ----
    loop {
        if buf.len() >= 8 {
            let l = i32::from_be_bytes([buf[0], buf[1], buf[2], buf[3]]) as usize;
            if buf.len() >= l { break; }
        }
        match stream.read(&mut tmp) {
            Ok(0) | Err(_) => return,
            Ok(n) => buf.extend_from_slice(&tmp[..n]),
        }
    }
    let code = i32::from_be_bytes([buf[4], buf[5], buf[6], buf[7]]);
    if code == 80877103 {
        // SSLRequest: no TLS here; reply 'N' and await the plain startup
        let _ = stream.write_all(b"N");
        buf.drain(..8);
        loop {
            if buf.len() >= 8 {
                let l = i32::from_be_bytes([buf[0], buf[1], buf[2], buf[3]]) as usize;
                if buf.len() >= l { break; }
            }
            match stream.read(&mut tmp) {
                Ok(0) | Err(_) => return,
                Ok(n) => buf.extend_from_slice(&tmp[..n]),
            }
        }
    }
    let len = i32::from_be_bytes([buf[0], buf[1], buf[2], buf[3]]) as usize;
    // startup is the framing exception: its length covers protocol + payload
    let protocol = i32::from_be_bytes([buf[4], buf[5], buf[6], buf[7]]);
    buf.drain(..len); // the WHOLE startup message goes; nothing may remain
    if protocol != 196608 {
        stream.write_all(&error_response("28000", "unsupported protocol")).ok();
        return;
    }
    let mut out = Vec::new();
    out.extend(authentication_ok());
    for (k, v) in [
        ("server_version", "16.9"),
        ("client_encoding", "UTF8"),
        ("DateStyle", "ISO, MDY"),
        ("integer_datetimes", "on"),
        ("TimeZone", "UTC"),
        ("standard_conforming_strings", "on"),
    ] {
        out.extend(parameter_status(k, v));
    }
    out.extend(ready_for_query(b'I'));
    if stream.write_all(&out).is_err() { return; }
    stream.flush().ok();

    let mut conn = Conn { statements: HashMap::new(), portals: HashMap::new(), in_tx: false };

    loop {
        // ---- one message: [tag][i32 len][body]; len counts its 4 bytes but
        // NOT the tag, so total wire size is 1 + len (startup was the exception)
        while buf.len() < 5 {
            match stream.read(&mut tmp) {
                Ok(0) | Err(_) => return,
                Ok(n) => buf.extend_from_slice(&tmp[..n]),
            }
        }
        let want = 1 + i32::from_be_bytes([buf[1], buf[2], buf[3], buf[4]]) as usize;
        while buf.len() < want {
            match stream.read(&mut tmp) {
                Ok(0) | Err(_) => return,
                Ok(n) => buf.extend_from_slice(&tmp[..n]),
            }
        }
        let tag = buf[0];
        let mlen = i32::from_be_bytes([buf[1], buf[2], buf[3], buf[4]]) as usize;
        let total = 1 + mlen;
        if mlen < 4 { continue; }
        let body = if mlen > 4 { buf[5..total].to_vec() } else { Vec::new() };
        buf.drain(..total);

        let mut reply: Vec<u8> = Vec::new();
        match tag {
            b'Q' => { // simple query
                let query = cstring_at(&body, 0);
                let mut d = db.lock().unwrap();
                let (payload, tx) = run_query(&mut d, &query, &[], conn.in_tx);
                conn.in_tx = tx;
                reply.extend(payload);
                reply.extend(ready_for_query(if tx { b'T' } else { b'I' }));
            }
            b'P' => { // Parse: name\0 sql\0 i16 nparams i32* oids
                let name = cstring_at(&body, 0);
                let sql = cstring_at(&body, name.len() + 1);
                conn.statements.insert(name, sql);
                reply.extend(parse_complete());
            }
            b'B' => { // Bind: portal\0 stmt\0 fmts, values, result-fmts
                let portal = cstring_at(&body, 0);
                let sstart = portal.len() + 1;
                let stmt = cstring_at(&body, sstart);
                let sql = conn.statements.get(&stmt).cloned().unwrap_or_default();
                // the body carries the statement NAME here, not the SQL text;
                // parameters start right after name\0
                let mut p = sstart + stmt.len() + 1;
                let mut params: Vec<Option<String>> = Vec::new();
                if p + 2 <= body.len() {
                    let nfmt = i16::from_be_bytes([body[p], body[p + 1]]) as usize;
                    p += 2 + nfmt * 2;
                    if p + 2 <= body.len() {
                        let nvals = i16::from_be_bytes([body[p], body[p + 1]]) as usize;
                        p += 2;
                        for _ in 0..nvals {
                            if p + 4 > body.len() { break; }
                            let vlen = i32::from_be_bytes([
                                body[p], body[p + 1], body[p + 2], body[p + 3]]);
                            p += 4;
                            if vlen < 0 {
                                params.push(None); // NULL
                            } else {
                                let end = (p + vlen as usize).min(body.len());
                                params.push(Some(String::from_utf8_lossy(&body[p..end]).into_owned()));
                                p = end;
                            }
                        }
                    }
                }
                conn.portals.insert(portal, (sql, params));
                reply.extend(bind_complete());
            }
            b'D' => { // Describe: kind('S'|'P') + name\0
                let kind = body[0];
                let name = cstring_at(&body, 1);
                let sql = match kind {
                    b'S' => conn.statements.get(&name).cloned().unwrap_or_default(),
                    _ => conn.portals.get(&name).map(|(s, _)| s.clone()).unwrap_or_default(),
                };
                reply.extend(msg(b't', &i16be(0))); // ParameterDescription
                if returns_rows(&sql) {
                    if sql.trim().to_lowercase() == "select 1" {
                        reply.extend(row_description(&[int4col("n")]));
                    } else if sql.to_lowercase().contains("count(*)") {
                        reply.extend(row_description(&[int4col("count")]));
                    } else {
                        reply.extend(row_description(&[int4col("id"), textcol("name")]));
                    }
                } else {
                    reply.extend(no_data());
                }
            }
            b'E' => { // Execute: portal\0 i32 maxrows
                let portal = cstring_at(&body, 0);
                let (sql, params) = conn.portals.get(&portal).cloned().unwrap_or_default();
                let mut d = db.lock().unwrap();
                let (payload, tx) = run_query(&mut d, &sql, &params, conn.in_tx);
                conn.in_tx = tx;
                reply.extend(payload);
            }
            b'S' => { // Sync: end of batch
                reply.extend(ready_for_query(if conn.in_tx { b'T' } else { b'I' }));
            }
            b'C' => { // Close statement/portal -> CloseComplete
                reply.extend(msg(b'3', &[]));
            }
            b'H' => { /* Flush: answers are already eager */ }
            b'X' => { // Terminate: an open tx rolls back on disconnect
                if conn.in_tx {
                    let mut d = db.lock().unwrap();
                    if let Some((rows, next)) = d.snapshot.take() {
                        d.rows = rows;
                        d.next_id = next;
                    }
                    d.tx_note("ROLLBACK(dropped)");
                }
                return;
            }
            _ => {
                reply.extend(error_response("08P01", "protocol violation"));
                reply.extend(ready_for_query(if conn.in_tx { b'T' } else { b'I' }));
            }
        }
        if reply.is_empty() { continue; }
        if stream.write_all(&reply).is_err() { return; }
        stream.flush().ok();
    }
}

/// Start the desk; return its port. Given: the whole chapter dials this.
fn start_pg_desk() -> u16 {
    let listener = std::net::TcpListener::bind("127.0.0.1:0").unwrap();
    let port = listener.local_addr().unwrap().port();
    let db: Shared = Arc::new(Mutex::new(Db { next_id: 1, ..Db::default() }));
    std::thread::spawn(move || {
        for stream in listener.incoming().flatten() {
            let db = Arc::clone(&db);
            std::thread::spawn(move || handle(stream, db));
        }
    });
    port
}

let pg_port = start_pg_desk(); // every later pool dials this port
println!("local postgres desk on 127.0.0.1:{} (protocol v3)", pg_port);


Worth reading once: the framing note in the comment is the whole protocol — `[tag][len][body]` where `len` counts its own four bytes but *not* the tag byte, except `StartupMessage` where the length covers everything (including the protocol version, which lives *inside* the length). `Parse`/`Bind`/`Describe`/`Execute` are per-connection state machines; `Sync` ends a batch with `ReadyForQuery`, whose status byte is what `sqlx` polls to know if a transaction is open. Transactions snapshot the table at `BEGIN` and restore on `ROLLBACK` — and a client that disconnects mid-transaction gets its work rolled back by the `Terminate` handler, which is the server-side behavior `&mut Transaction` in §5 relies on. The desk answers real error codes (`42P01` undefined table, `25P01` no transaction), which is why §10's error layer is honest.


## 1. Connecting: Options and the Pool

**`PgConnectOptions`** is the connection's configuration — host, port, user, database — built with chained methods, no URL parsing required (though `PgPoolOptions::new().connect("postgres://...")` accepts one). **`PgPool`** is the chapter's real subject: it opens connections lazily up to `max_connections`, hands them out to awaiting callers, returns them on drop, and is **cheap to clone** — the clone shares the pool, like chapter 18's multiplexed connection but with a real socket per checkout. Every `fetch_*` method takes `&pool` (or `&mut *tx` inside a transaction) precisely because the pool *decides* which connection serves the call. The type at the boundary is `PgRow`; everything you read out of it is decoded on demand.


In [ ]:
// 1.1 Options + pool; the pool connects lazily and clones freely.
// pool options come via the full path below
use sqlx::postgres::PgConnectOptions;

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1")
        .port(pg_port)
        .username("desk")
        .database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5)
        .acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts)
        .await
        .unwrap();

    // the clone shares the pool — pass it to tasks without wrapping in Arc
    let pool2 = pool.clone();
    drop(pool2); // closing a clone closes nothing

    let size = pool.size();
    println!("pool up: size={} (lazy — connections open on first use)", size);
    let _ = &pool; // the pool lives to the end of the cell, then closes
});


Three reads. **`connect_with` returns when the pool is ready**, but the connections are *lazy*: `size()` right after creation reports what opened during the handshake ping, and more open on demand up to the cap. **The two builder knobs that matter** are `max_connections` (§7's sizing) and `acquire_timeout` (how long a caller waits for a free connection before erroring — §7 times it). **Clone-then-drop is a no-op** on purpose: the pool's lifetime is shared, which is what makes it safe to hand to every task in a service without an `Arc`.


## 2. Queries: fetch_all, fetch_optional, fetch_one

`sqlx::query("...")` builds a runtime query; the three `fetch` shapes are the whole API surface. **`fetch_all`** returns `Vec<PgRow>` (zero rows = empty vec, never an error); **`fetch_optional`** returns `Option<PgRow>` — `None` for an *empty result*, and this is the honest detail the chapter teaches: a SQL **error still errors** (query `missing_table` and you get `Err`, not `None`); **`fetch_one`** demands exactly one row — empty is an `Err` (`RowNotFound`). Values decode through `row.get("col")` with the type inferred from the binding, or `row.try_get` when the schema might surprise you. The desk's `SELECT * FROM t WHERE name = $1` shows the `$1` placeholder — §6 makes it move.


In [ ]:
// 2.1 The three fetch shapes against seeded rows.
// get/try_get accessors: imported once here, persisted below (E0252 if repeated)
use sqlx::Row;
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5).acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts).await.unwrap();

    for name in ["ana", "bob", "cy"] {
        let r = sqlx::query("INSERT INTO t (name) VALUES ($1)")
            .bind(name)
            .execute(&pool)
            .await
            .unwrap();
        assert_eq!(r.rows_affected(), 1);
    }

    let all: Vec<sqlx::postgres::PgRow> =
        sqlx::query("SELECT * FROM t").fetch_all(&pool).await.unwrap();
    let ids: Vec<i32> = all.iter().map(|r| r.get("id")).collect();
    println!("fetch_all = {} rows, ids sorted = {:?}", all.len(), ids);

    let hit: Option<sqlx::postgres::PgRow> =
        sqlx::query("SELECT * FROM t").fetch_optional(&pool).await.unwrap();
    println!("fetch_optional on 3 rows = hit with id {:?}", hit.map(|r| r.get::<i32, _>("id")));

    let empty: Option<sqlx::postgres::PgRow> = sqlx::query("SELECT * FROM t WHERE name = $1")
        .bind("no-such-name")
        .fetch_optional(&pool)
        .await
        .unwrap();
    println!("fetch_optional on an empty RESULT = {:?} (not an error)", empty.is_none());

    let miss = sqlx::query("SELECT * FROM missing").fetch_optional(&pool).await;
    println!("fetch_optional on a SQL ERROR = Err({:?})",
        match &miss { Err(sqlx::Error::Database(e)) => e.code().unwrap_or_default().into_owned(), _ => "?".into() });

    let one: i32 = sqlx::query("SELECT 1").fetch_one(&pool).await.unwrap().get("n");
    println!("fetch_one scalar = {}", one);
});


Three reads. **`rows_affected()`** is the write-side receipt: `execute` returns how many rows the statement touched — the `INSERT 0 1` command tag from the wire, made data. **Empty-result vs SQL-error is a two-line distinction** with real consequences: `fetch_optional` is for "the row may legitimately not exist" (`WHERE email = ?` lookups), never for "the query might be malformed." **`fetch_one` on `SELECT 1`** shows the scalar shape — one row, named column, `get` decodes — which is how `count(*)` receipts read in §5.


## 3. Rows to Structs: FromRow

`PgRow` is the raw contract; your struct is the typed one, and **`FromRow` is the bridge**. `sqlx::query_as::<_, Item>("SELECT * FROM t")` decodes each row through `Item: FromRow` — and with the `macros` feature off, **you write the impl by hand**: `row.try_get("id")?` per field, `?` folding decode errors into the query's `sqlx::Error`. This is chapter 17's serde-derive story inverted: the derive (`#[derive(sqlx::FromRow)]`) generates exactly these `try_get` calls; writing two by hand shows you what the macro was hiding. The rule the compiler enforces: **every column you `try_get` must exist in the result set, and its type must convert** — `i32` from `int4`, `Option<String>` from nullable `text` (§4).


In [ ]:
// 3.1 The hand-written FromRow, then query_as decodes through it.
#[derive(Debug, PartialEq)]
struct Fill {
    id: i32,
    name: Option<String>,
}

impl<'r> sqlx::FromRow<'r, sqlx::postgres::PgRow> for Fill {
    fn from_row(row: &'r sqlx::postgres::PgRow) -> Result<Self, sqlx::Error> {
        Ok(Fill {
            id: row.try_get("id")?,      // int4 -> i32 or Err on mismatch
            name: row.try_get("name")?,  // nullable text -> Option<String>
        })
    }
}

let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5).acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts).await.unwrap();

    let fills: Vec<Fill> = sqlx::query_as("SELECT * FROM t")
        .fetch_all(&pool)
        .await
        .unwrap();
    let sorted: Vec<i32> = fills.iter().map(|f| f.id).collect();
    println!("query_as decoded {} Fill rows, ids {:?}", fills.len(), sorted);

    // the same rows, decoded ad hoc: row.get with a turbofish-free binding
    let first = sqlx::query("SELECT * FROM t").fetch_one(&pool).await.unwrap();
    let (id, name): (i32, Option<String>) = (first.get("id"), first.get("name"));
    println!("ad-hoc decode: id={} name={:?}", id, name);
});


Two reads. **The `?`s in `from_row` are the decode contract**: `try_get` fails when the column is absent (the `SELECT` stopped including it) or the type refuses (asking for `i64` from `int4` is allowed — `decode` coerces where lossless — but asking for `String` from an `int4` column is a `ColumnDecode` error). **`query_as` is generic over the destination**, which is why the binding carries the type: `let fills: Vec<Fill> = query_as(...)` — the same "you choose `T`" move as §18.2's `get::<Option<String>>`, now struct-shaped.


## 4. NULL and Option

The schema's nullability is part of its type, and **`Option` is the only honest Rust translation**. The desk's `name` column is nullable, so `Fill.name` is `Option<String>`: a `NULL` on the wire (length −1, no bytes) decodes to `None`, a value to `Some`. Writing follows the same rule — `bind(Option::<String>::None)` stores a `NULL`. What the type system forbids: decoding a `NULL` into bare `String` is a **runtime decode error** (`UnexpectedNullError`), not a panic and not an empty string — the schema said *maybe absent*, so the type must say it too. This is chapter 8's `Option` discipline enforced by a database.


In [ ]:
// 4.1 Insert a NULL, read it back as None; the wrong type refuses.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5).acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts).await.unwrap();

    sqlx::query("INSERT INTO t (name) VALUES ($1)")
        .bind(Option::<String>::None) // an absent value, not ""
        .execute(&pool)
        .await
        .unwrap();

    let rows: Vec<Fill> = sqlx::query_as("SELECT * FROM t")
        .fetch_all(&pool)
        .await
        .unwrap();
    let null_row = rows.iter().find(|f| f.name.is_none()).expect("the NULL row is there");
    println!("NULL decoded to None: fill id={} name={:?}", null_row.id, null_row.name);

    // the refusal: decoding NULL into a bare String is a decode ERROR
    let bad: Result<Vec<(i32, String)>, _> =
        sqlx::query_as("SELECT id, name FROM t").fetch_all(&pool).await;
    match bad {
        Err(sqlx::Error::ColumnDecode { source, .. }) => {
            println!("bare String vs NULL -> ColumnDecode({})", source);
        }
        other => println!("unexpected: {:?}", other.is_ok()),
    }
});


Two reads. **`bind(Option::<String>::None)`** needs the turbofish because `None` alone has no element type — the same inference limit as chapter 9's `None::<i32>`; the wire then carries length −1, which is §0.2's `NULL`. **The `ColumnDecode` refusal is the schema speaking**: had the column been declared `NOT NULL`, the bare `String` would be *correct* — the type error is not pedantry, it is the difference between "this can be absent" and "this cannot."


## 5. Transactions Across Await Points

`pool.begin()` returns a **`Transaction`** — a value that owns a pooled connection and marks it "in transaction" (`ReadyForQuery`'s status byte flips to `T` on the wire). Three exits: **`.commit().await`** makes the work permanent; **`tx.rollback().await`** undoes it; **dropping the `Transaction` without committing issues the rollback for you** — sqlx sends it while returning the connection to the pool, which is what makes `?`-propagation safe: early-return from a function holding `tx` and nothing partial survives. And the chapter-16 headline applies verbatim: **the transaction holds its connection across every `.await` inside the block** — queries run with `&mut *tx`, so the borrow checker can see the transaction is in use; you cannot accidentally interleave two tasks on one transaction because *owning* it is exclusive.


In [ ]:
// 5.1 Commit, drop-rollback, and explicit rollback — with count receipts.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5).acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts).await.unwrap();
    async fn count(pool: &sqlx::PgPool) -> i32 {
        let c: i32 = sqlx::query("SELECT count(*) FROM t")
            .fetch_one(pool).await.unwrap().get("count");
        c
    }

    let before = count(&pool).await;

    // commit path: the insert survives
    {
        let mut tx = pool.begin().await.unwrap();
        sqlx::query("INSERT INTO t (name) VALUES ($1)")
            .bind("tx-kept")
            .execute(&mut *tx) // &mut *tx: borrow the transaction's connection
            .await
            .unwrap();
        tx.commit().await.unwrap();
    }
    let after_commit = count(&pool).await;
    assert_eq!(after_commit, before + 1);

    // drop path: the insert vanishes — the drop issued ROLLBACK for us
    {
        let mut tx = pool.begin().await.unwrap();
        sqlx::query("INSERT INTO t (name) VALUES ($1)")
            .bind("tx-ghost")
            .execute(&mut *tx)
            .await
            .unwrap();
        drop(tx); // <- the whole lesson: no commit call anywhere
        tokio::time::sleep(std::time::Duration::from_millis(250)).await; // let it land
    }
    let after_drop = count(&pool).await;
    assert_eq!(after_drop, before + 1, "the ghost row was rolled back");

    // explicit rollback (rollback takes self by value; no mut binding needed)
    {
        let tx = pool.begin().await.unwrap();
        tx.rollback().await.unwrap();
    }
    assert_eq!(count(&pool).await, before + 1);
    println!("counts: before={} +1 committed ={} (drop-rollback kept it {})",
        before, after_commit, after_drop);
});


Three reads. **`&mut *tx` is the idiom and it is precise**: `Transaction` derefs to the connection, so `&mut *tx` borrows the connection *through* the transaction — the query runs on that connection inside that transaction, and the borrow ends before `.commit()` needs `tx` by value. **The 250ms sleep is honest instrumentation**, not flakiness: the drop-rollback is issued asynchronously when the `Transaction` drops; a real service just never reads until its own next statement, which arrives *after* the rollback on the same connection's queue. **The desk implements this for real**: §0.2 snapshots the table at `BEGIN` and restores on rollback — the count assertions prove server-side transactional semantics, not client-side bookkeeping.


## 6. Bind Parameters and SQL Injection

Every query so far carried its values through **`$1, $2, ...` placeholders plus `.bind(...)`** — the extended-protocol way of sending data *alongside* the statement, never spliced into it. The security lesson is one line: a value that travels as a bound parameter **cannot change the statement's shape**, so the classic injection `name = ''; DROP TABLE t; --` is just a weird string looking for a name. The desk demonstrates: bind the whole hostile string, and the row that comes back is the string itself. String formatting into SQL is the anti-pattern this replaces — and the type system helps only if you route values through `bind`, which is why every query in the chapter does.


In [ ]:
// 6.1 The hostile string, bound and harmless.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5).acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts).await.unwrap();

    let hostile = "x'; DROP TABLE t; --";
    sqlx::query("INSERT INTO t (name) VALUES ($1)")
        .bind(hostile) // the whole string is DATA
        .execute(&pool)
        .await
        .unwrap();

    let found: Option<sqlx::postgres::PgRow> = sqlx::query("SELECT * FROM t WHERE name = $1")
        .bind(hostile)
        .fetch_optional(&pool)
        .await
        .unwrap();
    let stored: String = found.unwrap().get("name");
    assert_eq!(stored, hostile);
    // the table is intact: the count includes the hostile row
    let n: i32 = sqlx::query("SELECT count(*) FROM t").fetch_one(&pool).await.unwrap().get("count");
    println!("hostile string stored verbatim ({} chars); table alive with {} rows",
        stored.len(), n);
});


Two reads. **On the wire, the value never touched the SQL**: `Parse` carried `INSERT INTO t (name) VALUES ($1)` and `Bind` carried `"x'; DROP TABLE t; --"` as a length-prefixed parameter — §0.2's `Bind` parser reads it as bytes, exactly the boundary real servers enforce. **`WHERE name = $1` is also the exact-match idiom**: §2's empty-result lookup and this injection demo are the same query shape — parameters are for filters, inserts, and updates alike, and there is no case in this chapter where a value is formatted into SQL text.


## 7. The Pool: Sizing, Acquire Timeouts

Two pool knobs are production decisions. **`max_connections`** caps how many sockets the pool opens — sized to what the *server* tolerances and the *workload's* concurrency actually need, not to core counts; every checkout beyond the cap waits. **`acquire_timeout`** bounds that wait: under saturation, `pool.acquire()` (or any `fetch_*`) errors with `PoolTimedOut` instead of queueing forever — measured here by a 1-connection pool holding its only socket past a 200ms budget. The debugging habit: `pool.size()` is *open connections*, not *in use*; saturation shows up as timing out acquires while `size()` sits at the cap.


In [ ]:
// 7.1 A 1-connection pool, one held checkout, one timed-out acquire.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let tiny = sqlx::postgres::PgPoolOptions::new()
        .max_connections(1)
        .acquire_timeout(std::time::Duration::from_millis(200))
        .connect_with(opts)
        .await
        .unwrap();

    let held = tiny.acquire().await.unwrap(); // the only socket, held
    let t0 = std::time::Instant::now();
    let second = tiny.acquire().await; // must wait: cap reached
    let waited = t0.elapsed();
    println!("second acquire: timed_out={} after={}ms",
        second.is_err(), waited.as_millis());
    match second {
        Err(sqlx::Error::PoolTimedOut) => println!("error flavor = PoolTimedOut (acquire_timeout fired)"),
        _ => println!("unexpected outcome"),
    }
    drop(held); // returned to the pool; the next acquire succeeds immediately
    let third = tiny.acquire().await;
    println!("after releasing the held checkout: acquired_ok={}", third.is_ok());
});


Two reads. **`PoolTimedOut` is a *backpressure* signal, not a bug**: it says "your workload wants more concurrency than the pool allows" — the fix is sizing (`max_connections`) or making the work smaller, never disabling the timeout. **The desk needed nothing special for this**: sockets are real (`TcpStream`s from §0.2), so holding a checkout is holding a socket, and the pool's accounting is client-side — which is why this receipt transfers unchanged to a real server.


## 8. Migrations: Schema as Code

Real teams change schemas incrementally, and **migrations are ordered SQL files with recorded versions** — `0001_init.sql` creates `t`, `0002_add_note.sql` alters it; a `migrations` table records which are applied; startup applies the pending ones in order, each *in a transaction*. The chapter's teaching version is the same contract in miniature: a `Vec<(&str, &str)>` of (version, sql), a loop that skips applied versions and executes new ones — **each inside `pool.begin()`/`commit()`**, so a failing migration leaves no half-schema. The `sqlx::migrate!` macro embeds a real `migrations/` directory into the binary with the same semantics (§9), and `sqlx-cli` scaffolds the files; the loop here is what that machinery does underneath.


In [ ]:
// 8.1 A migrations table, applied in order, each in a transaction.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5).acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts).await.unwrap();

    // the desk's migration ledger: (version, description, applied?)
    // (a real deployment keeps this in a _migrations table; the desk's shared
    // in-memory state stands in for it — re-running this cell sees it applied)
    static APPLIED: std::sync::Mutex<Vec<&'static str>> = std::sync::Mutex::new(Vec::new());

    let migrations: &[(&'static str, &str)] = &[
        ("0001_init", "the fills table exists"),
        ("0002_indexes", "an index plan for lookups"),
    ];
    for (version, _desc) in migrations {
        let mut applied = APPLIED.lock().unwrap();
        if applied.contains(version) {
            println!("{}: already applied, skipping", version);
            continue;
        }
        // the schema work itself, transactional like the real runner
        let mut tx = pool.begin().await.unwrap();
        sqlx::query("SELECT 1").execute(&mut *tx).await.unwrap();
        tx.commit().await.unwrap();
        applied.push(version);
        println!("{}: applied in a transaction", version);
    }
    // idempotence receipt: run the loop again, nothing re-applies
    for (version, _) in migrations {
        let applied = APPLIED.lock().unwrap();
        assert!(applied.contains(version));
    }
    println!("second pass: all versions already applied (idempotent)");
});


Two reads. **Ordering is the whole design**: `0001` before `0002` forever, applied exactly once — which is why the ledger, not the files, is the source of truth for "what did we run." **Each migration in its own transaction** is the safety property: a migration that fails partway rolls back to the last good schema, and the runner stops — never half-applied, never silently continuing. (The desk's `SELECT 1` stands in for DDL because the teaching table's shape is fixed; a real `0001_init.sql` contains the `CREATE TABLE`.)


## 9. sqlx in the Ecosystem: Macros, tokio-postgres, Diesel

Three ways to say the same `SELECT`, and the choice is about *when* things are checked. **The sqlx macros** (`query!`, `query_as!`) read your SQL **at compile time** against a live `DATABASE_URL`, return generated row structs, and fail the build on a bad column — compile-time schema checking, at the cost of needing a reachable database whenever you compile (and `offline mode` + a committed `sqlx-data.json` to soften that). **This chapter's runtime API** checks at run time: same types, same pool, same transactions; the SQL is a string and mistakes are `Err`s (§10's catalogue exists because of that). **tokio-postgres** is the lower-level async driver (sqlx itself speaks the wire protocol; tokio-postgres is its own implementation) — more control, fewer batteries. **Diesel** is the ORM end: schemas as Rust types, queries built by combinators, compile-time checking without a database — at the price of learning its DSL. The chapter teaches sqlx-runtime because its types *are* sqlx-macros' types: everything here transfers.


In [ ]:
// 9.1 The same query, runtime vs macro — the macro form shown in a comment.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5).acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts).await.unwrap();

    // runtime form (this chapter): SQL is data, decode is explicit
    let rows: Vec<Fill> = sqlx::query_as("SELECT * FROM t")
        .fetch_all(&pool)
        .await
        .unwrap();

    // macro form (needs the `macros` feature + DATABASE_URL at compile time):
    // let rows = sqlx::query_as!(Fill, "SELECT id, name FROM t")
    //     .fetch_all(&pool)
    //     .await
    //     .unwrap();
    // — identical call chain; the macro checked the SQL and generated the
    //   FromRow-equivalent code while `cargo build` ran.

    println!("runtime decode: {} rows (the macro form would compile-check this SQL against the live schema)", rows.len());
});


One read: **the call sites differ by one `!`**. That is the whole comparison — and the reason this chapter's skills are not throwaway: switching to macros later changes the *checking time*, not a single type or method. The trade to remember: macros catch schema drift at `cargo build`; runtime catches it in tests — the same positioning argument as §17.10's blocking-vs-async table, decided by when you want to find out.


## 10. Reading the Compiler: E-Codes

`sqlx::Error` has three layers, and the *shape* of the error tells you which layer failed before you read a word of the message. Plus the rustc codes the chapter's code provokes.

| Diagnostic | The story it tells |
|---|---|
| **`RowNotFound`** | `fetch_one` met an empty result — the one-call fix is `fetch_optional` when absence is legal. |
| **`ColumnDecode`** | the row had the column but the type refused (bare `String` vs `NULL` is the classic) — the schema said nullable, so the field is `Option`. |
| **`Database(code)`** | the server rejected the SQL: **`42P01`** undefined table, **`42601`** syntax error, **`25P01`** no transaction in progress, **`25001`** already in one — `err.code()` is the machine-readable layer (PostgreSQL's SQLSTATE), the message is for humans. |
| **`PoolTimedOut`** | no connection became free within `acquire_timeout` — backpressure, not breakage; §7's sizing conversation. |
| **`Protocol`** | client and server disagree about the wire (sqlx's own guard: `expected ReadyForQuery, got X`) — in this chapter it means the desk misbehaved; in production it means a proxy or version mismatch. |
| **E0277** | the future is not `Send` — a `Transaction` held across `.await` inside a `tokio::spawn` task that also holds a non-Send guard; the fix is scoping the transaction (§5) or `spawn_blocking` (§16). |
| **E0507/E0382** | the `Transaction` was moved while the query still borrowed `&mut *tx` — end the borrow (the `.await` returns) before `commit()` consumes `tx`. |


In [ ]:
// 10.1 The three error layers, demonstrated on the desk.
let rt = tokio::runtime::Runtime::new().unwrap();
rt.block_on(async {
    let opts = PgConnectOptions::new()
        .host("127.0.0.1").port(pg_port).username("desk").database("desk");
    let pool = sqlx::postgres::PgPoolOptions::new()
        .max_connections(5).acquire_timeout(std::time::Duration::from_secs(5))
        .connect_with(opts).await.unwrap();

    // layer 1: RowNotFound — fetch_one on an empty result
    let miss = sqlx::query("SELECT * FROM t WHERE name = $1")
        .bind("nobody").fetch_one(&pool).await;
    println!("empty result via fetch_one: {:?}", miss.unwrap_err());

    // layer 2: Database with a code — the server rejected the SQL
    let bad = sqlx::query("SELECT * FROM missing").fetch_one(&pool).await;
    match bad {
        Err(sqlx::Error::Database(e)) => {
            println!("server refused: code={} message={}", e.code().unwrap_or_default(), e.message());
        }
        _ => unreachable!(),
    }

    // layer 3: ColumnDecode — the row is fine, the type refuses
    let bad_decode = sqlx::query_as::<_, (i32, String)>("SELECT id, name FROM t")
        .fetch_all(&pool)
        .await;
    match bad_decode {
        Err(e) => println!("decode refused NULL for String: {:?}...", e),
        Ok(v) => println!("decoded {} (desk has no NULL here? re-run after 4.1)", v.len()),
    }
});


Two reads. **Matching on the enum variant *is* the error-handling API**: `if let Err(sqlx::Error::Database(e)) = ...` and then `e.code()` — no string parsing, the layering is in the types. **The decode error only fires when a NULL is present**, which is why 10.1's third probe depends on §4.1 having run: restart the kernel, run 4.1, then 10.1 — the notebook is sequential on purpose, like the schema it teaches.


## 11. Mental Model: The Receipts

| Claim | Receipt |
|---|---|
| Options are the config; the pool is the handle | §1.1: `PgConnectOptions` chain, `PgPoolOptions` knobs, `connect_with` |
| A pool clone is shared, not a copy | §1.1: clone dropped, pool unaffected |
| Writes report rows affected | §2.1: `rows_affected() == 1` per insert |
| Empty result ≠ SQL error | §2.1: `fetch_optional` → `None` on empty, `Err(42P01)` on a bad table |
| `fetch_one` demands a row | §2.1/§10.1: `RowNotFound` on empty |
| `FromRow` is the struct contract | §3.1: `query_as` decodes `Vec<Fill>` through two `try_get`s |
| NULL is `Option`, enforced | §4.1: `None` round-trips; bare `String` refuses (`ColumnDecode`) |
| Transactions survive awaits | §5.1: `&mut *tx` queries, then `commit`/drop/rollback with count receipts |
| Drop is rollback | §5.1: ghost row gone after `drop(tx)` — server-side snapshot restored |
| Bound values are data, never SQL text | §6.1: the injection string stored verbatim, table intact |
| Pools saturate and time out | §7.1: `PoolTimedOut` at the cap, instant acquire after release |
| Migrations are ordered, transactional, once | §8.1: applied/skip ledger, idempotent second pass |
| Macros move the check to build time | §9.1: same call chain, one `!` of difference |


## 12. Cheat Sheet

```rust
// ---- connect: options in, pool out (clone the pool freely) ----
let opts = PgConnectOptions::new()
    .host("127.0.0.1").port(5432).username("app").database("desk");
let pool = sqlx::postgres::PgPoolOptions::new()
    .max_connections(10)
    .acquire_timeout(std::time::Duration::from_secs(3))
    .connect_with(opts).await?;              // or .connect("postgres://...")

// ---- the three fetch shapes (runtime SQL; decode at the call site) ----
let all: Vec<PgRow> = sqlx::query("SELECT * FROM t").fetch_all(&pool).await?;
let one: Option<PgRow> = sqlx::query("SELECT ...").fetch_optional(&pool).await?;
                                                       // None = empty result ONLY
let row: PgRow = sqlx::query("SELECT ...").fetch_one(&pool).await?; // Err if empty
let n = one.unwrap().get::<i32, _>("id");    // decode: int4 -> i32
let n = r.rows_affected();                   // write receipt (execute)

// ---- rows to structs: FromRow (hand-written = what the derive generates) ----
struct Fill { id: i32, name: Option<String> }   // nullable column = Option
impl<'r> sqlx::FromRow<'r, PgRow> for Fill {
    fn from_row(row: &'r PgRow) -> Result<Self, sqlx::Error> {
        Ok(Fill { id: row.try_get("id")?, name: row.try_get("name")? })
    }
}
let fills: Vec<Fill> = sqlx::query_as("SELECT * FROM t").fetch_all(&pool).await?;

// ---- NULL: bind Options, decode Options, never bare types ----
sqlx::query("INSERT INTO t (name) VALUES ($1)")
    .bind(Option::<String>::None).execute(&pool).await?;  // NULL on the wire
// decoding NULL into String = Err(ColumnDecode) — the schema said nullable

// ---- transactions survive await points ----
let mut tx = pool.begin().await?;            // ReadyForQuery flips to 'T'
sqlx::query("INSERT ...").bind(&v).execute(&mut *tx).await?;  // &mut *tx
tx.commit().await?;                          // permanent
// drop(tx)                                  // = ROLLBACK issued for you
tx.rollback().await?;                        // explicit undo

// ---- values are ALWAYS bound, never formatted into SQL ----
sqlx::query("SELECT * FROM t WHERE name = $1").bind(user_input)...

// ---- pool sizing is a production decision ----
.max_connections(10)                         // cap: callers beyond it wait
.acquire_timeout(Duration::from_secs(3))     // wait bound: PoolTimedOut on fire

// ---- migrations: ordered, applied once, each in a transaction ----
// sqlx::migrate!("migrations").run(&pool).await?;  // real deployment form
```

Seven load-bearing sentences: **the pool is configured once and cloned everywhere; every query's rows decode into your types at the call site; `fetch_optional` converts only empty *results*, never errors; a nullable column is `Option` in the struct or the decode refuses; the transaction is a value whose drop is a rollback; values travel as bound parameters and never touch the SQL text; pool sizing and acquire timeouts are the backpressure knobs.**


## 13. Review

Twelve questions, in chapter order. Answers are in the prose above each.

1. **(§0)** What is the v3 wire framing rule — what does the length field count, and which message is the exception?
2. **(§1)** What does `pool.clone()` share, and why is no `Arc` needed to hand a pool to every task?
3. **(§2)** Distinguish the three `fetch` shapes on an empty result: what does each return or error with?
4. **(§2)** What does `rows_affected()` report, and which wire receipt is it derived from?
5. **(§3)** What does `try_get` check on each field, and which two failures can a hand-written `FromRow` hit that the compiler cannot see?
6. **(§4)** Why is decoding a `NULL` into `String` an `Err` rather than an empty string, and what would make `String` correct?
7. **(§5)** What does `&mut *tx` borrow through, and why can two tasks never share one transaction accidentally?
8. **(§5)** What happens to an in-flight transaction when its `Transaction` value is dropped — who issues the rollback, and what does the desk snapshot prove?
9. **(§6)** Why can a bound parameter not change a statement's shape, and where on the wire do the value and the SQL travel?
10. **(§7)** What does `PoolTimedOut` tell you about the workload, and which two settings govern it?
11. **(§8)** What are the three properties of a migration runner, and what does "each migration in a transaction" buy when one fails?
12. **(§9)** Where do the sqlx runtime API and the macros differ, and where do they agree?

### What Comes Next

| Next | What it borrows from here |
|---|---|
| **20 — PyO3** | the escape hatch back to Python: boundary structs become `#[pyclass]`es; the decode-at-the-boundary discipline is the export. |
| **21 — Capstone Prep** | error handling across crate boundaries is §10's layering generalized; benchmarking measures §7's pool knobs. |
| **22 — Capstone** | §5's transactions plus §18's cache in front of §17's feed is the market-data service: persist fills with `&mut *tx`, read-through cache, graceful shutdown closes pools. |

Carry forward: **the pool is the handle and clones are free; rows decode into your types or they refuse; the transaction is a value and its drop is a rollback; NULL is Option all the way down.**
